In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 6 — NETWORK CONSTRUCTION
# Inputs:  outputs/02_returns_matrix.parquet
#          outputs/06_tail_dependence_lower.parquet
#          outputs/04_correlation_spearman_full.parquet
#          outputs/10_cointegration_results.csv        ← optional
# Outputs: outputs/12_glasso_network.graphml
#          outputs/13_mst_tail_dependence.graphml
#          outputs/phase6_glasso_partial_corr.parquet
#          outputs/phase6_communities.csv
#          outputs/phase6_node_metrics.csv
# ═══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.covariance import GraphicalLassoCV, GraphicalLasso
from sklearn.preprocessing import StandardScaler
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.sparse import csr_matrix
from statsmodels.stats.multitest import multipletests

# Network
try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    HAS_NX = False

# Community detection
try:
    import community as community_louvain
    HAS_LOUVAIN = True
except ImportError:
    HAS_LOUVAIN = False

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

Path("outputs").mkdir(exist_ok=True)
Path("outputs/figures").mkdir(exist_ok=True)


# ═══════════════════════════════════════════════════════════════════════════════
# INSTALL MISSING PACKAGES
# ═══════════════════════════════════════════════════════════════════════════════
import subprocess, sys

if not HAS_NX:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "networkx", "-q"])
    import networkx as nx
    print("  ✅ networkx installed")

if not HAS_LOUVAIN:
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "python-louvain", "-q"])
    import community as community_louvain
    print("  ✅ python-louvain installed")


# ═══════════════════════════════════════════════════════════════════════════════
# LOAD INPUTS
# ═══════════════════════════════════════════════════════════════════════════════
print("Loading inputs...")

log_returns  = pd.read_parquet("outputs/02_returns_matrix.parquet")
lam_L_mat    = pd.read_parquet("outputs/06_tail_dependence_lower.parquet")
spearman_mat = pd.read_parquet("outputs/04_correlation_spearman_full.parquet")
coin_meta    = pd.read_csv("outputs/03_coin_metadata.csv")

# Optional Phase 5
try:
    coint_df  = pd.read_csv("outputs/10_cointegration_results.csv")
    has_phase5 = True
    print(f"  Phase 5 cointegration results loaded : {len(coint_df)} pairs")
except FileNotFoundError:
    coint_df   = pd.DataFrame()
    has_phase5 = False
    print("  Phase 5 not available — proceeding without cointegration edges")

coins = log_returns.columns.tolist()
N     = len(coins)
print(f"  Coins         : {N}")
print(f"  Return matrix : {log_returns.shape}")


# ═══════════════════════════════════════════════════════════════════════════════
# 6.1  GLASSO — PARTIAL CORRELATION NETWORK
# ═══════════════════════════════════════════════════════════════════════════════
# GLASSO fits an L1-regularised precision matrix (inverse covariance).
# A non-zero entry (i,j) in the precision matrix means coin i and coin j
# are directly related AFTER controlling for all other coins.
#
# Why this matters: in Phase 2 we computed raw pairwise correlation.
# But A and C might look correlated simply because both correlate with B.
# GLASSO removes these indirect connections — only DIRECT relationships survive.
# This gives us a much cleaner picture of genuine structural similarity.
#
# The regularisation parameter λ is tuned by cross-validation (GraphicalLassoCV).
# Higher λ = sparser network (fewer edges). We let CV pick the optimal sparsity.

print("\n── Phase 6.1 │ GLASSO Partial Correlation Network ─────────────────────────")

# Build complete return matrix — fill NaN with column mean for GLASSO
# (GLASSO requires complete data; mean-filling late-start coins is acceptable
#  since GLASSO is regularised and robust to small imputation errors)
returns_filled = log_returns.copy()
for col in returns_filled.columns:
    col_mean = returns_filled[col].mean()
    returns_filled[col] = returns_filled[col].fillna(col_mean)

# Standardise — GLASSO works on correlation scale
scaler  = StandardScaler()
X_scaled= scaler.fit_transform(returns_filled.values)

print("  Fitting GraphicalLassoCV (cross-validated regularisation)...")
print("  This may take 2–5 minutes...")

try:
    glasso_cv = GraphicalLassoCV(
        cv=5,
        n_refinements=4,
        max_iter=200,
        tol=1e-3,
        n_jobs=-1,
        verbose=False,
    )
    glasso_cv.fit(X_scaled)
    best_alpha  = glasso_cv.alpha_
    precision   = glasso_cv.precision_
    covariance  = glasso_cv.covariance_
    print(f"  CV-selected regularisation α : {best_alpha:.6f}")

except Exception as e:
    print(f"  ⚠️  GraphicalLassoCV failed ({e}) — falling back to α=0.1")
    best_alpha = 0.1
    glasso_cv  = GraphicalLasso(alpha=best_alpha, max_iter=500, tol=1e-3)
    glasso_cv.fit(X_scaled)
    precision  = glasso_cv.precision_
    covariance = glasso_cv.covariance_

# ─── Convert precision matrix to partial correlations ─────────────────────────
# Partial correlation = -precision(i,j) / sqrt(precision(i,i) * precision(j,j))
diag_sqrt  = np.sqrt(np.diag(precision))
denom      = np.outer(diag_sqrt, diag_sqrt)
partial_corr = -precision / denom
np.fill_diagonal(partial_corr, 1.0)

partial_corr_df = pd.DataFrame(partial_corr, index=coins, columns=coins)

# ─── Network statistics ────────────────────────────────────────────────────────
upper_tri = partial_corr[np.triu_indices(N, k=1)]
nonzero   = np.sum(np.abs(upper_tri) > 1e-4)
density   = nonzero / len(upper_tri)

print(f"\n  Partial correlation network:")
print(f"    Non-zero edges         : {nonzero:,} / {len(upper_tri):,}  ({density:.1%} density)")
print(f"    Mean |partial corr|    : {np.mean(np.abs(upper_tri)):.4f}")
print(f"    Max  |partial corr|    : {np.max(np.abs(upper_tri)):.4f}")

# ─── Build NetworkX graph ─────────────────────────────────────────────────────
G_glasso = nx.Graph()
G_glasso.add_nodes_from(coins)

edge_threshold = 1e-4   # only include edges with meaningful partial correlation
for i in range(N):
    for j in range(i + 1, N):
        pc = partial_corr[i, j]
        if abs(pc) > edge_threshold:
            G_glasso.add_edge(coins[i], coins[j],
                              weight=float(pc),
                              abs_weight=float(abs(pc)))

print(f"    NetworkX edges added   : {G_glasso.number_of_edges():,}")
print(f"    Connected components   : {nx.number_connected_components(G_glasso)}")


# ─── Node-level metrics ───────────────────────────────────────────────────────
degree_centrality   = nx.degree_centrality(G_glasso)
betweenness         = nx.betweenness_centrality(G_glasso, weight="abs_weight",
                                                normalized=True)
# Strength = sum of absolute partial correlations for each node
strength = {coin: sum(abs(d["weight"])
                      for _, _, d in G_glasso.edges(coin, data=True))
            for coin in coins}

node_metrics = pd.DataFrame({
    "coin_id"            : coins,
    "degree_centrality"  : [degree_centrality.get(c, 0) for c in coins],
    "betweenness"        : [betweenness.get(c, 0) for c in coins],
    "strength"           : [strength.get(c, 0) for c in coins],
})

print(f"\n  Most central coins (GLASSO strength — most connected after removing indirect links):")
print(node_metrics.sort_values("strength", ascending=False)
      .head(10)[["coin_id", "strength", "degree_centrality"]]
      .to_string(index=False))

print(f"\n  Most peripheral coins (lowest strength — most independent):")
print(node_metrics.sort_values("strength")
      .head(10)[["coin_id", "strength", "degree_centrality"]]
      .to_string(index=False))


# ═══════════════════════════════════════════════════════════════════════════════
# 6.2  MINIMUM SPANNING TREE ON λL (CRASH SIMILARITY)
# ═══════════════════════════════════════════════════════════════════════════════
# The MST finds the minimal set of edges that connects all N coins,
# where edge weight = λL (crash tail dependence from Phase 3).
# Higher λL = stronger crash similarity = we WANT this edge in the tree.
# So we use (1 - λL) as the distance for MST construction.
#
# The MST reveals the BACKBONE of crash-time redundancy:
#   - Coins in the core (high degree) = substitutable with many others
#   - Coins at the periphery (degree = 1) = most unique, least redundant
#   - Clusters of coins connected by short edges = crash together as a group

print("\n── Phase 6.2 │ Minimum Spanning Tree on λL ─────────────────────────────────")

# Build distance matrix from λL — use (1 - λL) as distance
# Handle NaN: replace with maximum distance (1.0)
lam_L_vals = lam_L_mat.values.copy()
np.fill_diagonal(lam_L_vals, 1.0)                    # diagonal = 1 → distance 0
lam_L_vals = np.where(np.isnan(lam_L_vals), 0.0, lam_L_vals)

distance_lL   = 1.0 - lam_L_vals
np.fill_diagonal(distance_lL, 0.0)
distance_lL   = np.clip(distance_lL, 0, 1)
distance_lL   = (distance_lL + distance_lL.T) / 2   # enforce symmetry

# Compute MST via scipy
sparse_dist   = csr_matrix(distance_lL)
mst_sparse    = minimum_spanning_tree(sparse_dist)
mst_arr       = mst_sparse.toarray()

# Build NetworkX MST graph
G_mst = nx.Graph()
G_mst.add_nodes_from(coins)

mst_edges = []
rows, cols = np.where(mst_arr > 0)
for r, c in zip(rows, cols):
    dist   = mst_arr[r, c]
    lam_l  = 1.0 - dist
    G_mst.add_edge(coins[r], coins[c],
                   weight=float(lam_l),
                   distance=float(dist))
    mst_edges.append((coins[r], coins[c], lam_l))

print(f"  MST edges          : {G_mst.number_of_edges()}")
print(f"  MST nodes          : {G_mst.number_of_nodes()}")
print(f"  Expected edges     : {N - 1}  (spanning tree)")

# ─── MST node degree distribution ────────────────────────────────────────────
mst_degrees   = dict(G_mst.degree())
degree_series = pd.Series(mst_degrees)

print(f"\n  MST degree distribution:")
for d in sorted(degree_series.unique()):
    cnt = (degree_series == d).sum()
    print(f"    Degree {d:2d} : {cnt:3d} coins")

# Hub coins — high degree in MST = most substitutable
hub_coins = degree_series.sort_values(ascending=False).head(10)
print(f"\n  Hub coins (core — most substitutable):")
for coin, deg in hub_coins.items():
    print(f"    {coin:30s} degree = {deg}")

# Peripheral coins — degree 1 = most unique
leaf_coins = degree_series[degree_series == 1].index.tolist()
print(f"\n  Leaf coins (periphery — most unique) : {len(leaf_coins)}")
print(f"    {', '.join(leaf_coins[:15])}{'...' if len(leaf_coins) > 15 else ''}")

# ─── Strongest MST edges (highest λL) ────────────────────────────────────────
mst_edge_df = pd.DataFrame(mst_edges, columns=["coin_a", "coin_b", "lambda_L"])
print(f"\n  Strongest MST edges (highest crash similarity on backbone):")
print(mst_edge_df.sort_values("lambda_L", ascending=False)
      .head(15).to_string(index=False))


# ═══════════════════════════════════════════════════════════════════════════════
# 6.3  COMMUNITY DETECTION — LOUVAIN ON BOTH NETWORKS
# ═══════════════════════════════════════════════════════════════════════════════
# Louvain algorithm finds communities by maximising modularity —
# groups of coins that are more densely connected to each other
# than to the rest of the network.
#
# We run it on both networks separately:
#   GLASSO communities  → groups sharing direct statistical relationships
#   MST communities     → groups that crash together as a unit
#
# Where both agree → HIGH CONFIDENCE cluster
# Where they disagree → flag for manual review

print("\n── Phase 6.3 │ Community Detection (Louvain) ───────────────────────────────")

# ─── GLASSO communities ───────────────────────────────────────────────────────
# Use absolute partial correlation as edge weight
G_glasso_abs = nx.Graph()
G_glasso_abs.add_nodes_from(coins)
for u, v, d in G_glasso.edges(data=True):
    G_glasso_abs.add_edge(u, v, weight=d["abs_weight"])

glasso_communities = community_louvain.best_partition(
    G_glasso_abs, weight="weight", random_state=42
)
n_glasso_comm = len(set(glasso_communities.values()))
print(f"\n  GLASSO communities    : {n_glasso_comm}")

# ─── MST communities ─────────────────────────────────────────────────────────
mst_communities = community_louvain.best_partition(
    G_mst, weight="weight", random_state=42
)
n_mst_comm = len(set(mst_communities.values()))
print(f"  MST (λL) communities  : {n_mst_comm}")

# ─── Build community dataframe ────────────────────────────────────────────────
community_df = pd.DataFrame({
    "coin_id"          : coins,
    "glasso_community" : [glasso_communities.get(c, -1) for c in coins],
    "mst_community"    : [mst_communities.get(c, -1) for c in coins],
})

# ─── Agreement check ──────────────────────────────────────────────────────────
# For each pair of coins in the same GLASSO community, check if they're also
# in the same MST community. Agreement rate = confidence of the cluster.

agreement_count = 0
total_same_glasso = 0

for comm_id in set(glasso_communities.values()):
    members = [c for c, g in glasso_communities.items() if g == comm_id]
    for i in range(len(members)):
        for j in range(i + 1, len(members)):
            total_same_glasso += 1
            if mst_communities.get(members[i]) == mst_communities.get(members[j]):
                agreement_count += 1

agreement_rate = agreement_count / max(total_same_glasso, 1)
print(f"\n  Cross-network agreement rate : {agreement_rate:.1%}")
print(f"  (pairs in same GLASSO community that are also in same MST community)")

if agreement_rate >= 0.70:
    print(f"  ✅ High agreement — communities are robust")
elif agreement_rate >= 0.50:
    print(f"  ⚠️  Moderate agreement — some clusters need review")
else:
    print(f"  ⚠️  Low agreement — GLASSO and MST see different structure")

# ─── Community size distributions ─────────────────────────────────────────────
print(f"\n  GLASSO community sizes:")
glasso_sizes = community_df.groupby("glasso_community")["coin_id"].count().sort_values(ascending=False)
for cid, size in glasso_sizes.items():
    members = community_df.loc[community_df["glasso_community"] == cid, "coin_id"].tolist()
    print(f"    Community {cid:2d} : {size:3d} coins  │ "
          f"{', '.join(members[:5])}{'...' if size > 5 else ''}")

print(f"\n  MST community sizes:")
mst_sizes = community_df.groupby("mst_community")["coin_id"].count().sort_values(ascending=False)
for cid, size in mst_sizes.items():
    members = community_df.loc[community_df["mst_community"] == cid, "coin_id"].tolist()
    print(f"    Community {cid:2d} : {size:3d} coins  │ "
          f"{', '.join(members[:5])}{'...' if size > 5 else ''}")

# ─── Flag disagreeing coins ───────────────────────────────────────────────────
# A coin "disagrees" if its GLASSO community neighbours are NOT its MST
# community neighbours. These coins sit on cluster boundaries.
disagree_coins = []
for coin in coins:
    glasso_comm = glasso_communities.get(coin)
    mst_comm    = mst_communities.get(coin)

    glasso_neighbours = [c for c, g in glasso_communities.items()
                         if g == glasso_comm and c != coin]
    mst_neighbours    = [c for c, m in mst_communities.items()
                         if m == mst_comm and c != coin]

    overlap = len(set(glasso_neighbours) & set(mst_neighbours))
    total   = len(set(glasso_neighbours) | set(mst_neighbours))
    jaccard = overlap / max(total, 1)

    if jaccard < 0.30:   # less than 30% overlap → disagreement
        disagree_coins.append(coin)

community_df["community_agreement"] = community_df["coin_id"].apply(
    lambda c: "disagree" if c in disagree_coins else "agree"
)

print(f"\n  Coins flagged for manual review (community disagreement) : {len(disagree_coins)}")
if disagree_coins:
    print(f"    {', '.join(disagree_coins)}")


# ═══════════════════════════════════════════════════════════════════════════════
# 6.4  ENRICH NODE METRICS WITH ALL SIGNALS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 6.4 │ Enriched Node Metrics ──────────────────────────────────────")

# MST degree per coin
mst_degree_df = pd.DataFrame({
    "coin_id"    : list(mst_degrees.keys()),
    "mst_degree" : list(mst_degrees.values()),
})

# Average λL per coin (how crash-similar is this coin to all others on average)
avg_lam_L = lam_L_mat.replace(1.0, np.nan).mean(axis=1)
avg_lam_L_df = avg_lam_L.reset_index()
avg_lam_L_df.columns = ["coin_id", "avg_lambda_L"]

# Merge everything
node_metrics = (
    node_metrics
    .merge(mst_degree_df,  on="coin_id", how="left")
    .merge(avg_lam_L_df,   on="coin_id", how="left")
    .merge(community_df,   on="coin_id", how="left")
    .merge(coin_meta[["coin_id", "median_daily_vol", "amihud_ratio"]],
           on="coin_id", how="left")
)

# Redundancy flag: high strength + high mst_degree + high avg_lambda_L
node_metrics["network_redundancy"] = (
    (node_metrics["strength"]      > node_metrics["strength"].quantile(0.75)) &
    (node_metrics["mst_degree"]    > 2) &
    (node_metrics["avg_lambda_L"]  > node_metrics["avg_lambda_L"].quantile(0.75))
)

n_redundant_nodes = node_metrics["network_redundancy"].sum()
print(f"  Coins flagged as network-redundant : {n_redundant_nodes}")
print(f"\n  Most redundant coins (network perspective):")
print(node_metrics[node_metrics["network_redundancy"]]
      .sort_values("avg_lambda_L", ascending=False)
      .head(15)[["coin_id", "mst_degree", "strength", "avg_lambda_L"]]
      .to_string(index=False))


# ═══════════════════════════════════════════════════════════════════════════════
# EXPORT GRAPHML FILES
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Exporting network files ─────────────────────────────────────────────────")

# Annotate GLASSO graph with node attributes
for coin in coins:
    row = node_metrics.set_index("coin_id").loc[coin] \
          if coin in node_metrics["coin_id"].values else {}
    G_glasso.nodes[coin]["glasso_community"] = int(glasso_communities.get(coin, -1))
    G_glasso.nodes[coin]["mst_community"]    = int(mst_communities.get(coin, -1))
    G_glasso.nodes[coin]["strength"]         = float(strength.get(coin, 0))
    G_glasso.nodes[coin]["avg_lambda_L"]     = float(avg_lam_L.get(coin, 0))

# Annotate MST graph
for coin in coins:
    G_mst.nodes[coin]["glasso_community"] = int(glasso_communities.get(coin, -1))
    G_mst.nodes[coin]["mst_community"]    = int(mst_communities.get(coin, -1))
    G_mst.nodes[coin]["mst_degree"]       = int(mst_degrees.get(coin, 0))
    G_mst.nodes[coin]["avg_lambda_L"]     = float(avg_lam_L.get(coin, 0))

nx.write_graphml(G_glasso, "outputs/12_glasso_network.graphml")
print("  ✅ 12_glasso_network.graphml   → partial correlation network")

nx.write_graphml(G_mst, "outputs/13_mst_tail_dependence.graphml")
print("  ✅ 13_mst_tail_dependence.graphml → λL minimum spanning tree")


# ═══════════════════════════════════════════════════════════════════════════════
# SAVE TABULAR OUTPUTS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Saving Phase 6 outputs ──────────────────────────────────────────────────")

partial_corr_df.to_parquet("outputs/phase6_glasso_partial_corr.parquet")
print("  ✅ phase6_glasso_partial_corr.parquet → N×N partial correlation matrix")

community_df.to_csv("outputs/phase6_communities.csv", index=False)
print("  ✅ phase6_communities.csv             → GLASSO + MST community per coin")

node_metrics.to_csv("outputs/phase6_node_metrics.csv", index=False)
print("  ✅ phase6_node_metrics.csv            → centrality + redundancy per coin")

mst_edge_df.to_csv("outputs/phase6_mst_edges.csv", index=False)
print("  ✅ phase6_mst_edges.csv               → MST edge list with λL weights")

print(f"""
╔══════════════════════════════════════════════════════════════════╗
║  PHASE 6 COMPLETE                                                ║
╠══════════════════════════════════════════════════════════════════╣
║  GLASSO network edges       : {G_glasso.number_of_edges():,}                          ║
║  GLASSO communities         : {n_glasso_comm}                              ║
║  MST communities            : {n_mst_comm}                              ║
║  Cross-network agreement    : {agreement_rate:.1%}                         ║
║  Network-redundant coins    : {n_redundant_nodes}                             ║
║  Coins flagged for review   : {len(disagree_coins)}                             ║
╠══════════════════════════════════════════════════════════════════╣
║  Next: Phase 7 — Composite Similarity Score                     ║
║  Key inputs for Phase 7:                                         ║
║    phase6_glasso_partial_corr.parquet                            ║
║    phase6_communities.csv                                        ║
║    phase6_node_metrics.csv                                       ║
║    All Phase 2–5 similarity signals                              ║
╚══════════════════════════════════════════════════════════════════╝
""")